<div style="border-left: 5px solid #4285f4; padding: 16px 20px; margin: 6px 0 20px 0; border-radius: 6px; background: rgba(66,133,244,0.06);">
<h1>✨ Mejorar imágenes con IA</h1>
<p><strong>Restauración de fotografías con CodeFormer y HYPIR</strong></p>
<p>Un flujo sencillo para recuperar detalles y ampliar tus imágenes con dos motores de restauración.</p>
<p><strong>Perfil natural:</strong> HYPIR guarda también un resultado mezclado con la imagen original ampliada para reducir artefactos generativos. Los pesos y la caché se conservan en Google Drive para reutilizarlos entre sesiones.</p>
<hr>
<h3>🧭 Cómo usar este cuaderno</h3>
<ol>
<li>⚙️ Preparar el entorno.</li>
<li>📤 Subir una foto o seleccionar una que ya tengas guardada.</li>
<li>🎚️ Elegir el modelo, la ampliación y los ajustes disponibles.</li>
<li>🚀 Restaurar la imagen.</li>
<li>🖼️ Comparar el antes y el después.</li>
<li>📥 Descargar el resultado.</li>
</ol>
<p><strong>💡 Recomendación:</strong> selecciona una GPU en <em>Entorno de ejecución → Cambiar tipo de entorno de ejecución</em> para acelerar el procesamiento. Si Colab reinicia el entorno, vuelve a ejecutar la preparación.</p>
</div>

> ****Versión 5 · caché de inferencia verificada:** identifica las imágenes por su contenido y comprueba los parámetros y las dimensiones antes de reutilizar resultados. Los modelos y los archivos se conservan en Google Drive cuando está montado.


<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>⚙️ 01 · Preparar el entorno</h2>
</div>

Ejecuta esta celda primero. Prepara las dependencias, conecta Google Drive y prepara CodeFormer y HYPIR, y descarga sus pesos. HYPIR también descargará el modelo base Stable Diffusion 2.1 durante su primera ejecución; esto puede tardar y ocupar varios GB.

<blockquote>⏳ La primera ejecución puede tardar unos minutos mientras se preparan los componentes.</blockquote>

El código fuente y la caché maestra del modelo base se conservan en Drive. Para acelerar la carga, SD 2.1 se copia al SSD temporal de Colab una vez por sesión; los pesos se verifican antes de reutilizarlos. La primera preparación necesita conexión para descargar los componentes que falten.


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, urllib.request, time, zipfile, hashlib, importlib.metadata, json
from concurrent.futures import ThreadPoolExecutor, as_completed

INICIO_PREPARACION = time.perf_counter()
TIEMPOS_PREPARACION = {}
from PIL import Image, ImageOps
from IPython.display import display, Markdown
import matplotlib.pyplot as plt

PROYECTO = Path('/content/Mejorar_Imagenes_con_IA')
ENTRADA = PROYECTO / 'entrada'
RESULTADOS = PROYECTO / 'resultados'
EXTENSIONES = {
    '.png', '.jpg', '.jpeg', '.webp',
    '.bmp', '.tif', '.tiff',
}

# Montar Google Drive si estamos en Colab.
EN_COLAB = False
files = None
try:
    from google.colab import drive, files
    drive.mount('/content/drive', force_remount=False)
    DRIVE_PROYECTO = Path(
        '/content/drive/MyDrive/Mejorar_Imagenes_con_IA'
    )
    DRIVE_ORIGINALES = DRIVE_PROYECTO / 'originales'
    DRIVE_RESULTADOS = DRIVE_PROYECTO / 'resultados'
    DRIVE_MODELOS = DRIVE_PROYECTO / 'modelos'
    DRIVE_CACHE_HF = DRIVE_PROYECTO / 'cache' / 'huggingface'
    for carpeta in (
        DRIVE_ORIGINALES, DRIVE_RESULTADOS, DRIVE_MODELOS, DRIVE_CACHE_HF
    ):
        carpeta.mkdir(parents=True, exist_ok=True)
    # Mantiene el caché de Hugging Face entre sesiones de Colab.
    # El almacenamiento en Drive evita volver a descargar SD 2.1,
    # aunque la primera carga desde Drive puede ser algo más lenta.
    os.environ['HF_HOME'] = str(DRIVE_CACHE_HF)
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(DRIVE_CACHE_HF / 'hub')  # compatibilidad
    os.environ['HF_HUB_CACHE'] = str(DRIVE_CACHE_HF / 'hub')
    EN_COLAB = True
except Exception as e:
    print('Aviso de Google Drive/Colab:', repr(e))

for carpeta in (ENTRADA, RESULTADOS):
    carpeta.mkdir(parents=True, exist_ok=True)

# Buscar un repositorio ya preparado antes de descargar uno nuevo.
# Si no existe una copia válida, descargar la rama predeterminada oficial.
candidatos_repo = [
    (DRIVE_PROYECTO / 'CodeFormer') if EN_COLAB else PROYECTO / 'CodeFormer',
    PROYECTO / 'CodeFormer',
    Path('/content/CodeFormer'),
]
CODEFORMER = next(
    (p for p in candidatos_repo
     if (p / 'inference_codeformer.py').is_file()),
    None,
)

if CODEFORMER is None:
    # En Colab, guardar también el repositorio y sus pesos en Drive
    # para que no haya que descargarlos de nuevo en cada sesión.
    CODEFORMER = (
        DRIVE_PROYECTO / 'CodeFormer'
        if EN_COLAB else PROYECTO / 'CodeFormer'
    )
    if CODEFORMER.exists():
        shutil.rmtree(CODEFORMER)
    print('Descargando el repositorio oficial de CodeFormer desde la rama predeterminada…')
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/sczhou/CodeFormer.git',
         str(CODEFORMER)],
        check=True,
    )
else:
    print('Repositorio encontrado:', CODEFORMER)

# Instalar/comprobar dependencias tanto en un clon nuevo como en uno existente.
# En Colab, los paquetes del entorno no persisten aunque el repositorio esté en Drive.
archivo_requisitos = CODEFORMER / 'requirements.txt'
if not archivo_requisitos.is_file():
    raise FileNotFoundError(
        'No se encontró requirements.txt en el repositorio: '
        + str(archivo_requisitos)
    )
# No repetir pip install al volver a ejecutar esta celda en la MISMA sesión.
# El marcador vive en /content, no en Drive: Colab lo pierde al reiniciar el runtime,
# por lo que una sesión nueva sigue instalando las dependencias que necesita.
HASH_REQUIREMENTS = hashlib.sha256(archivo_requisitos.read_bytes()).hexdigest()
MARCADOR_CODEFORMER_DEPS = Path(
    f'/content/.mejorar_codeformer_deps_py{sys.version_info.major}{sys.version_info.minor}_{HASH_REQUIREMENTS[:12]}'
)
inicio_instalacion_codeformer = time.perf_counter()
if MARCADOR_CODEFORMER_DEPS.is_file():
    print('Dependencias CodeFormer ya instaladas en esta sesión; se omite pip install.')
else:
    print('Instalando dependencias CodeFormer para esta sesión…')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-r', str(archivo_requisitos)],
        check=True,
    )
    MARCADOR_CODEFORMER_DEPS.write_text(
        f'requirements_sha256={HASH_REQUIREMENTS}\n', encoding='utf-8'
    )
TIEMPOS_PREPARACION['Dependencias CodeFormer (instalar/comprobar)'] = time.perf_counter() - inicio_instalacion_codeformer

SCRIPT = CODEFORMER / 'inference_codeformer.py'
if not SCRIPT.is_file():
    mensaje = 'No se encontró inference_codeformer.py en '
    raise FileNotFoundError(mensaje + str(CODEFORMER))

# Algunas copias no incluyen version.py.
# BasicSR lo importa al iniciar.
version_py = CODEFORMER / 'basicsr' / 'version.py'
if (CODEFORMER / 'basicsr').is_dir() and not version_py.exists():
    version_py.write_text(
        "__version__ = '0.0.0-local'\n__gitsha__ = 'unknown'\n",
        encoding='utf-8',
    )

# Descargar pesos desde nuestro propio GitHub Release.
# Se verifica SHA-256 para detectar descargas incompletas o archivos alterados.
import hashlib
inicio_descarga_pesos_codeformer = time.perf_counter()

BASE_URL = (
    'https://github.com/Pablitus666/Mejorar_Imagenes_con_IA/'
    'releases/download/modelos-v1/'
)

PESO_CODEFORMER = (
    CODEFORMER / 'weights' / 'CodeFormer' / 'codeformer.pth'
)
PESO_DETECCION = (
    CODEFORMER / 'weights' / 'facelib'
    / 'detection_Resnet50_Final.pth'
)
PESO_PARSING = CODEFORMER / 'weights' / 'facelib' / 'parsing_parsenet.pth'
PESO_YOLO = CODEFORMER / 'weights' / 'facelib' / 'yolov5l-face.pth'

PESOS = [
    {
        'nombre': 'codeformer.pth',
        'url': BASE_URL + 'codeformer.pth',
        'destino': PESO_CODEFORMER,
        'minimo_bytes': 100_000_000,
        'sha256': '1009e537e0c2a07d4cabce6355f53cb66767cd4b4297ec7a4a64ca4b8a5684b7',
    },
    {
        'nombre': 'detection_Resnet50_Final.pth',
        'url': BASE_URL + 'detection_Resnet50_Final.pth',
        'destino': PESO_DETECCION,
        'minimo_bytes': 1_000_000,
        'sha256': '6d1de9c2944f2ccddca5f5e010ea5ae64a39845a86311af6fdf30841b0a5a16d',
    },
    {
        'nombre': 'parsing_parsenet.pth',
        'url': BASE_URL + 'parsing_parsenet.pth',
        'destino': PESO_PARSING,
        'minimo_bytes': 1_000_000,
        'sha256': '3d558d8d0e42c20224f13cf5a29c79eba2d59913419f945545d8cf7b72920de2',
    },
    {
        'nombre': 'RealESRGAN_x2plus.pth',
        'url': BASE_URL + 'RealESRGAN_x2plus.pth',
        'destino': CODEFORMER / 'weights' / 'realesrgan' / 'RealESRGAN_x2plus.pth',
        'minimo_bytes': 60_000_000,
        'sha256': '49fafd45f8fd7aa8d31ab2a22d14d91b536c34494a5cfe31eb5d89c2fa266abb',
    },
    {
        'nombre': 'yolov5l-face.pth',
        'url': BASE_URL + 'yolov5l-face.pth',
        'destino': PESO_YOLO,
        'minimo_bytes': 100_000_000,
        'sha256': '1ba9d2125fda4d823df5152b9fc2903c59aa76c0d3771e02bcf13a56a282cf96',
    },
]

def calcular_sha256(ruta):
    sha = hashlib.sha256()
    with open(ruta, 'rb') as archivo:
        for bloque in iter(lambda: archivo.read(4 * 1024 * 1024), b''):
            sha.update(bloque)
    return sha.hexdigest()


# Caché persistente de verificaciones: solo reutiliza un SHA-256 si coinciden
# la ruta, el tamaño, la fecha de modificación y el hash esperado. Si cambia
# cualquier metadato, se vuelve a leer el archivo completo.
CACHE_SHA_DIR = (
    DRIVE_PROYECTO / 'cache' / 'verificaciones_sha256'
    if EN_COLAB else PROYECTO / 'cache' / 'verificaciones_sha256'
)
CACHE_SHA_DIR.mkdir(parents=True, exist_ok=True)

def sha256_verificado_cacheable(ruta, sha256_esperado):
    ruta = Path(ruta).resolve()
    stat = ruta.stat()
    clave = hashlib.sha256(str(ruta).encode('utf-8')).hexdigest()[:24]
    marcador = CACHE_SHA_DIR / f'{clave}.json'
    firma = {
        'ruta': str(ruta),
        'size': stat.st_size,
        'mtime_ns': stat.st_mtime_ns,
        'sha256_esperado': sha256_esperado.lower(),
    }
    try:
        datos = json.loads(marcador.read_text(encoding='utf-8'))
        if all(datos.get(k) == v for k, v in firma.items()) and datos.get('sha256'):
            return datos['sha256']
    except (OSError, ValueError, TypeError):
        pass
    digest = calcular_sha256(ruta).lower()
    if digest == sha256_esperado.lower():
        firma['sha256'] = digest
        temporal = marcador.with_suffix('.tmp')
        temporal.write_text(json.dumps(firma), encoding='utf-8')
        temporal.replace(marcador)
    return digest

def descargar_peso(url, destino, minimo_bytes, sha256_esperado):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)

    if destino.is_file() and destino.stat().st_size >= minimo_bytes:
        print(f'Verificando peso existente: {destino.name}…')
        sha_existente = sha256_verificado_cacheable(destino, sha256_esperado)
        if sha_existente.lower() == sha256_esperado.lower():
            print(f'Peso verificado: {destino.name} (SHA-256 correcto)')
            return
        print(f'El SHA-256 de {destino.name} no coincide; se descargará de nuevo.')

    temporal = destino.with_suffix(destino.suffix + '.part')
    if temporal.exists():
        temporal.unlink()
    print(f'Descargando {destino.name} desde nuestro Release…')
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=90) as respuesta, open(temporal, 'wb') as salida:
            total = int(respuesta.headers.get('Content-Length', '0') or 0)
            descargado = 0
            while True:
                bloque = respuesta.read(1024 * 1024)
                if not bloque:
                    break
                salida.write(bloque)
                descargado += len(bloque)
                if total:
                    print(f'  {descargado / total:.0%}', end='\r')

        tamano = temporal.stat().st_size
        if tamano < minimo_bytes:
            raise RuntimeError(
                f'El archivo descargado es demasiado pequeño ({tamano:,} bytes).'
            )
        sha_descargado = calcular_sha256(temporal)
        if sha_descargado.lower() != sha256_esperado.lower():
            raise RuntimeError(
                'La verificación SHA-256 ha fallado. '
                f'Esperado: {sha256_esperado}; obtenido: {sha_descargado}'
            )
        temporal.replace(destino)
        print(f'Verificado y guardado: {destino.name} ({tamano:,} bytes)')
    except Exception as e:
        if temporal.exists():
            temporal.unlink()
        raise RuntimeError(
            f'No se pudo descargar/verificar {destino.name} desde {url}. Detalle: {e}'
        ) from e

for peso in PESOS:
    descargar_peso(
        peso['url'], peso['destino'], peso['minimo_bytes'], peso['sha256']
    )

# Configurar CodeFormer para usar el peso RealESRGAN local.
# La URL oficial puede variar según la revisión del script (CodeFormer o Real-ESRGAN).
# Sustituimos cualquier URL HTTP(S) que termine en RealESRGAN_x2plus.pth,
# sin depender de la versión del dominio ni de la función que la contiene.
import re

PESO_REALESRGAN = (
    CODEFORMER / 'weights' / 'realesrgan' / 'RealESRGAN_x2plus.pth'
).resolve()
ruta_local_script = str(PESO_REALESRGAN)
contenido_script = SCRIPT.read_text(encoding='utf-8')

# Guardar una copia del script original una sola vez.
copia_seguridad = SCRIPT.with_suffix(SCRIPT.suffix + '.bak')
if not copia_seguridad.exists():
    shutil.copy2(SCRIPT, copia_seguridad)

# Ejemplo de URL oficial que se corrige:
# https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth
patron_url_realesrgan = re.compile(
    r"https?://[^\s\"']*RealESRGAN_x2plus\.pth",
    flags=re.IGNORECASE,
)
contenido_corregido, reemplazos = patron_url_realesrgan.subn(
    lambda _m: ruta_local_script,
    contenido_script,
)

if reemplazos:
    SCRIPT.write_text(contenido_corregido, encoding='utf-8')
    print(
        'URL externa de RealESRGAN sustituida por la ruta local '
        f'({reemplazos} reemplazo(s)).'
    )
elif ruta_local_script in contenido_script:
    print('CodeFormer ya estaba configurado para usar el RealESRGAN local.')
else:
    lineas_relevantes = [
        f'{n}: {linea.strip()}'
        for n, linea in enumerate(contenido_script.splitlines(), 1)
        if 'RealESRGAN' in linea or 'model_path' in linea
    ]
    raise RuntimeError(
        'No se encontró la URL de RealESRGAN_x2plus.pth en el script. '
        'No se modificó el script. Líneas relacionadas: '
        + (' | '.join(lineas_relevantes) or '(ninguna)')
    )

# Verificar que la URL externa ya no aparece y que el archivo local sí.
contenido_final = SCRIPT.read_text(encoding='utf-8')
if ruta_local_script not in contenido_final:
    raise RuntimeError(
        'La verificación de la ruta local de RealESRGAN falló: ' + str(SCRIPT)
    )
if patron_url_realesrgan.search(contenido_final):
    raise RuntimeError(
        'Todavía queda una URL externa de RealESRGAN_x2plus.pth en ' + str(SCRIPT)
    )
if not PESO_REALESRGAN.is_file() or PESO_REALESRGAN.stat().st_size == 0:
    raise FileNotFoundError(
        'No existe el peso local de RealESRGAN o está vacío: '
        + str(PESO_REALESRGAN)
    )

# Garantizar que desmarcar "Mejorar también el fondo" desactive
# RealESRGAN incluso si se reutiliza una copia cuyo valor predeterminado
# lo activa. La celda de restauración solo añade --bg_upsampler realesrgan
# cuando la opción está marcada.
patron_argumento_fondo = re.compile(
    r"(?m)^(?P<prefijo>\s*parser\.add_argument\(\s*['\"]--bg_upsampler['\"][^\n]*?\bdefault\s*=\s*)"
    r"['\"](?P<valor>[^'\"]*)['\"]"
)
coincidencia_fondo = patron_argumento_fondo.search(contenido_final)
if coincidencia_fondo and coincidencia_fondo.group('valor').lower() == 'realesrgan':
    contenido_final = patron_argumento_fondo.sub(
        lambda m: m.group('prefijo') + "'None'",
        contenido_final,
        count=1,
    )
    SCRIPT.write_text(contenido_final, encoding='utf-8')
    print('Control de fondo ajustado: RealESRGAN queda desactivado por defecto.')
elif coincidencia_fondo:
    print(
        'Control de fondo verificado: valor predeterminado =',
        repr(coincidencia_fondo.group('valor')),
    )
else:
    # Si no se declara un valor predeterminado explícito, argparse usa None.
    linea_bg = next(
        (linea.strip() for linea in contenido_final.splitlines()
         if '--bg_upsampler' in linea),
        None,
    )
    if linea_bg is None:
        raise RuntimeError(
            'No se encontró el argumento --bg_upsampler en CodeFormer; '
            'no es posible garantizar el control de mejora del fondo.'
        )
    if re.search(r"default\s*=\s*['\"]?realesrgan", linea_bg, re.IGNORECASE):
        raise RuntimeError(
            'CodeFormer activa RealESRGAN por defecto y no se pudo ajustar '
            'automáticamente el argumento --bg_upsampler.'
        )
    print('Control de fondo verificado en el argumento:', linea_bg)

print('Peso RealESRGAN local:', PESO_REALESRGAN)
print('Verificación SHA-256: ya realizada durante la descarga de pesos.')
TIEMPOS_PREPARACION['Pesos CodeFormer/Real-ESRGAN (comprobación/descarga)'] = time.perf_counter() - inicio_descarga_pesos_codeformer

# ------------------------------------------------------------
# Preparar HYPIR con código fuente cacheado en Drive.
# Se conserva el PyTorch de Colab y se fija una revisión conocida
# para que cambios futuros en la rama main no alteren el resultado.
# ------------------------------------------------------------
HYPIR_REVISION = 'b61d107'
HYPIR_REPO = PROYECTO / 'HYPIR'  # código extraído en disco local de Colab
HYPIR_SCRIPT = HYPIR_REPO / 'test.py'
HYPIR_REVISION_FILE = HYPIR_REPO / '.mejorar_hypir_revision'
HYPIR_PESO = (
    DRIVE_MODELOS / 'HYPIR_sd2.pth'
    if EN_COLAB else PROYECTO / 'modelos' / 'HYPIR_sd2.pth'
)
HYPIR_PESO.parent.mkdir(parents=True, exist_ok=True)
CACHE_CODIGO_HYPIR = (
    DRIVE_PROYECTO / 'cache' / 'codigo_hypir'
    if EN_COLAB else PROYECTO / 'cache' / 'codigo_hypir'
)
CACHE_CODIGO_HYPIR.mkdir(parents=True, exist_ok=True)
HYPIR_ARCHIVO = CACHE_CODIGO_HYPIR / f'HYPIR_{HYPIR_REVISION}.zip'
HYPIR_URL_ARCHIVO = (
    f'https://github.com/XPixelGroup/HYPIR/archive/{HYPIR_REVISION}.zip'
)

# La primera sesión descarga un ZIP pequeño y lo conserva en Drive.
# En sesiones posteriores se extrae localmente, evitando clonar GitHub otra vez.
def archivo_hypir_valido(ruta_zip):
    try:
        with zipfile.ZipFile(ruta_zip, 'r') as zf:
            nombres = zf.namelist()
            return (
                zf.testzip() is None
                and any(n.endswith('/test.py') for n in nombres)
                and any(n.endswith('/HYPIR/enhancer/sd2.py') for n in nombres)
            )
    except (OSError, zipfile.BadZipFile):
        return False

inicio_codigo_hypir = time.perf_counter()
if (not HYPIR_SCRIPT.is_file() or not HYPIR_REVISION_FILE.is_file()
        or HYPIR_REVISION_FILE.read_text(encoding='utf-8').strip() != HYPIR_REVISION):
    if not archivo_hypir_valido(HYPIR_ARCHIVO):
        if HYPIR_ARCHIVO.exists():
            HYPIR_ARCHIVO.unlink()
        temporal_zip = HYPIR_ARCHIVO.with_suffix('.zip.part')
        print(f'Descargando código HYPIR {HYPIR_REVISION} para guardarlo en caché…')
        req = urllib.request.Request(
            HYPIR_URL_ARCHIVO,
            headers={'User-Agent': 'Mejorar-Imagenes-con-IA'},
        )
        try:
            with urllib.request.urlopen(req, timeout=90) as respuesta, open(temporal_zip, 'wb') as salida:
                shutil.copyfileobj(respuesta, salida)
            if not archivo_hypir_valido(temporal_zip):
                raise RuntimeError('El ZIP de HYPIR descargado no pasó la verificación.')
            temporal_zip.replace(HYPIR_ARCHIVO)
        except Exception:
            if temporal_zip.exists():
                temporal_zip.unlink()
            raise
    else:
        print('Archivo fuente HYPIR verificado en caché:', HYPIR_ARCHIVO)

    # Extraer en el disco temporal de Colab para acelerar los imports de Python.
    carpeta_extraccion = PROYECTO / '_extraccion_hypir'
    if carpeta_extraccion.exists():
        shutil.rmtree(carpeta_extraccion)
    carpeta_extraccion.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(HYPIR_ARCHIVO, 'r') as zf:
        # Rechazar rutas extrañas antes de extraer el archivo.
        raiz = carpeta_extraccion.resolve()
        for miembro in zf.infolist():
            destino_miembro = (carpeta_extraccion / miembro.filename).resolve()
            if destino_miembro != raiz and raiz not in destino_miembro.parents:
                raise RuntimeError('El ZIP de HYPIR contiene una ruta no válida.')
        zf.extractall(carpeta_extraccion)
    carpetas_fuente = [p for p in carpeta_extraccion.iterdir() if p.is_dir()]
    raiz_fuente = next((p for p in carpetas_fuente if (p / 'test.py').is_file()), None)
    if raiz_fuente is None or not (raiz_fuente / 'HYPIR' / 'enhancer' / 'sd2.py').is_file():
        raise FileNotFoundError('La fuente extraída de HYPIR no tiene la estructura esperada.')
    if HYPIR_REPO.exists():
        shutil.rmtree(HYPIR_REPO)
    raiz_fuente.replace(HYPIR_REPO)
    HYPIR_REVISION_FILE.write_text(HYPIR_REVISION + '\n', encoding='utf-8')
    shutil.rmtree(carpeta_extraccion, ignore_errors=True)
else:
    print('Código HYPIR disponible en el entorno:', HYPIR_SCRIPT, '(revisión', HYPIR_REVISION + ')')
TIEMPOS_PREPARACION['Código HYPIR (caché/descarga/extracción)'] = time.perf_counter() - inicio_codigo_hypir

if not (HYPIR_REPO / 'LICENSE').is_file():
    print('Aviso: no se encontró el archivo LICENSE de HYPIR en la copia local.')

# Las versiones se fijan para reproducibilidad. Si ya están correctas en este
# runtime, no se vuelve a ejecutar pip (evita resolución e instalaciones repetidas).
DEPENDENCIAS_HYPIR = {
    'accelerate': '1.4.0',
    'diffusers': '0.32.2',
    'transformers': '4.49.0',
    'peft': '0.14.0',
    'einops': '0.8.1',
    'omegaconf': '2.3.0',
    'timm': '1.0.15',
    'huggingface-hub': '0.36.0',
}
def dependencias_hypir_correctas():
    try:
        return all(
            importlib.metadata.version(paquete) == version
            for paquete, version in DEPENDENCIAS_HYPIR.items()
        )
    except importlib.metadata.PackageNotFoundError:
        return False

inicio_instalacion_hypir = time.perf_counter()
MARCADOR_HYPIR_DEPS = Path(
    f'/content/.mejorar_hypir_deps_py{sys.version_info.major}{sys.version_info.minor}'
)
if dependencias_hypir_correctas():
    print('Dependencias HYPIR ya están en las versiones fijadas; se omite pip install.')
else:
    print('Instalando/comprobando dependencias HYPIR sin reemplazar PyTorch…')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'accelerate==1.4.0', 'diffusers==0.32.2',
        'transformers==4.49.0', 'peft==0.14.0',
        'einops==0.8.1', 'omegaconf==2.3.0', 'timm==1.0.15',
        'huggingface_hub==0.36.0',
    ], check=True)
    if not dependencias_hypir_correctas():
        raise RuntimeError('Las dependencias HYPIR no coinciden con las versiones fijadas tras instalarlas.')
    MARCADOR_HYPIR_DEPS.write_text('dependencias_verificadas\n', encoding='utf-8')
TIEMPOS_PREPARACION['Dependencias HYPIR (instalar/comprobar)'] = time.perf_counter() - inicio_instalacion_hypir

# El peso se fija a una revisión concreta y se verifica por SHA-256.
# Así se detectan archivos truncados o alterados antes de iniciar la inferencia.
HYPIR_PESO_SHA256 = 'd538a2cb925451fab1f75adfe715ac2b1c8bb12fa32a0851ba01be2347b354d6'
HYPIR_PESO_REVISION = '93e4cceb10df58bb5d2441159f6abd4cf56430ba'
inicio_peso_hypir = time.perf_counter()
peso_correcto = (
    HYPIR_PESO.is_file()
    and HYPIR_PESO.stat().st_size > 500_000_000
    and sha256_verificado_cacheable(HYPIR_PESO, HYPIR_PESO_SHA256).lower() == HYPIR_PESO_SHA256
)
if not peso_correcto:
    if HYPIR_PESO.exists():
        print('El peso HYPIR local falta o no supera SHA-256; se volverá a descargar.')
        HYPIR_PESO.unlink()
    print('Descargando los pesos HYPIR (aprox. 1 GB) desde una revisión fijada…')
    from huggingface_hub import hf_hub_download
    peso_cache = Path(hf_hub_download(
        repo_id='lxq007/HYPIR',
        filename='HYPIR_sd2.pth',
        revision=HYPIR_PESO_REVISION,
        local_dir=str(HYPIR_PESO.parent),
    ))
    if peso_cache.resolve() != HYPIR_PESO.resolve():
        shutil.copy2(peso_cache, HYPIR_PESO)
    if calcular_sha256(HYPIR_PESO).lower() != HYPIR_PESO_SHA256:
        raise RuntimeError('El peso HYPIR descargado no coincide con el SHA-256 esperado.')
    print('Peso HYPIR descargado y verificado por SHA-256.')
else:
    print('Peso HYPIR encontrado y verificado:', HYPIR_PESO)
TIEMPOS_PREPARACION['Peso HYPIR (SHA-256/comprobación/descarga)'] = time.perf_counter() - inicio_peso_hypir

def copiar_arbol_paralelo(origen, destino, trabajadores=4):
    """Copia un snapshot al SSD resolviendo enlaces y copiando archivos en paralelo."""
    origen, destino = Path(origen), Path(destino)
    archivos = []
    for ruta in origen.rglob('*'):
        relativa = ruta.relative_to(origen)
        if '.cache' in relativa.parts:
            continue
        destino_ruta = destino / relativa
        if ruta.is_dir():
            destino_ruta.mkdir(parents=True, exist_ok=True)
        elif ruta.is_file():
            destino_ruta.parent.mkdir(parents=True, exist_ok=True)
            archivos.append((ruta, destino_ruta))
    errores = []
    with ThreadPoolExecutor(max_workers=trabajadores) as pool:
        futuros = [pool.submit(shutil.copy2, origen_archivo, destino_archivo)
                   for origen_archivo, destino_archivo in archivos]
        for futuro in as_completed(futuros):
            try:
                futuro.result()
            except Exception as exc:
                errores.append(exc)
    if errores:
        raise RuntimeError(f'Fallaron {len(errores)} archivos durante la copia paralela: {errores[0]}')
    return len(archivos)


# Acelerar la carga de SD 2.1: conservar la caché maestra en Drive, pero cargar
# desde el SSD local de Colab. Se valida la estructura real del modelo, no solo
# la existencia de una carpeta o un marcador.
HYPIR_BASE_MODEL_ID = 'STHCD/stable-diffusion-2-1-base'
HYPIR_BASE_MODEL_LOCAL = PROYECTO / '_cache_local' / 'stable-diffusion-2-1-base'
HYPIR_BASE_MODEL_MARKER = HYPIR_BASE_MODEL_LOCAL / '.mejorar_cache_completa'
HYPIR_BASE_MODEL_LOCAL.parent.mkdir(parents=True, exist_ok=True)


def modelo_sd21_valido(raiz):
    """Comprueba configuración y pesos esenciales de un snapshot SD 2.1."""
    raiz = Path(raiz)
    configuraciones = (
        'model_index.json',
        'scheduler/scheduler_config.json',
        'text_encoder/config.json',
        'tokenizer/tokenizer_config.json',
        'unet/config.json',
        'vae/config.json',
    )
    if not raiz.is_dir() or not all((raiz / rel).is_file() for rel in configuraciones):
        return False
    # Evita aceptar punteros Git-LFS, archivos vacíos o snapshots sin pesos.
    tokenizer = raiz / 'tokenizer'
    if not ((tokenizer / 'tokenizer.json').is_file()
            or ((tokenizer / 'vocab.json').is_file() and (tokenizer / 'merges.txt').is_file())):
        return False
    for componente in ('text_encoder', 'unet', 'vae'):
        carpeta = raiz / componente
        pesos = [p for patron in ('*.safetensors', '*.bin') for p in carpeta.glob(patron)
                 if p.is_file() and p.stat().st_size > 1024]
        if not pesos:
            return False
    try:
        import json as _json
        with open(raiz / 'model_index.json', encoding='utf-8') as f:
            indice = _json.load(f)
        if not isinstance(indice, dict) or not indice:
            return False
    except (OSError, ValueError, TypeError):
        return False
    return True

inicio_cache_base = time.perf_counter()
if modelo_sd21_valido(HYPIR_BASE_MODEL_LOCAL):
    print('Modelo base SD 2.1 validado en el SSD local de esta sesión; se omiten descarga y copia.')
    if not HYPIR_BASE_MODEL_MARKER.is_file():
        HYPIR_BASE_MODEL_MARKER.write_text(
            f'repo={HYPIR_BASE_MODEL_ID}\n', encoding='utf-8'
        )
    TIEMPOS_PREPARACION['Modelo base: descarga/comprobación HF'] = 0.0
    TIEMPOS_PREPARACION['Modelo base: copia a SSD local'] = 0.0
else:
    from huggingface_hub import snapshot_download
    cache_hub = os.environ.get('HF_HUB_CACHE')
    ruta_snapshot_base = None
    inicio_busqueda_hf = time.perf_counter()
    try:
        ruta_candidata = Path(snapshot_download(
            repo_id=HYPIR_BASE_MODEL_ID,
            cache_dir=cache_hub,
            local_files_only=True,
        ))
        if not modelo_sd21_valido(ruta_candidata):
            raise FileNotFoundError('El snapshot de Drive no contiene todas las configuraciones y pesos SD 2.1 requeridos.')
        ruta_snapshot_base = ruta_candidata
        print('Snapshot SD 2.1 encontrado y validado en la caché persistente de Drive.')
        TIEMPOS_PREPARACION['Modelo base: descarga/comprobación HF'] = time.perf_counter() - inicio_busqueda_hf
    except Exception as e_cache:
        print('La caché SD 2.1 no está completa; se comprobará/descargará desde Hugging Face:', str(e_cache)[:250])
        inicio_descarga_base = time.perf_counter()
        ruta_snapshot_base = Path(snapshot_download(
            repo_id=HYPIR_BASE_MODEL_ID,
            cache_dir=cache_hub,
            local_files_only=False,
        ))
        TIEMPOS_PREPARACION['Modelo base: descarga/comprobación HF'] = time.perf_counter() - inicio_descarga_base
        if not modelo_sd21_valido(ruta_snapshot_base):
            raise FileNotFoundError(
                'Hugging Face devolvió un snapshot SD 2.1 incompleto: faltan configuraciones o pesos esenciales. '
                f'Ruta: {ruta_snapshot_base}'
            )

    staging_base = HYPIR_BASE_MODEL_LOCAL.parent / '_stable-diffusion-2-1-base.tmp'
    if staging_base.exists():
        shutil.rmtree(staging_base)
    inicio_copia_base = time.perf_counter()
    print('Copiando el modelo base desde la caché persistente al SSD local de Colab…')
    # Resolver los enlaces del caché HF a archivos normales en el SSD local.
    # Copiar archivos en paralelo para aprovechar mejor la transferencia desde Drive.
    cantidad_archivos = copiar_arbol_paralelo(ruta_snapshot_base, staging_base, trabajadores=4)
    print(f'Copia paralela completada: {cantidad_archivos} archivos.')
    if not modelo_sd21_valido(staging_base):
        shutil.rmtree(staging_base, ignore_errors=True)
        raise FileNotFoundError('La copia local de SD 2.1 no superó la validación de configuraciones y pesos.')
    if HYPIR_BASE_MODEL_LOCAL.exists():
        shutil.rmtree(HYPIR_BASE_MODEL_LOCAL)
    staging_base.rename(HYPIR_BASE_MODEL_LOCAL)
    HYPIR_BASE_MODEL_MARKER.write_text(
        f'repo={HYPIR_BASE_MODEL_ID}\n', encoding='utf-8'
    )
    TIEMPOS_PREPARACION['Modelo base: copia a SSD local'] = time.perf_counter() - inicio_copia_base
    print(f'Caché local SD 2.1 validada y preparada en {TIEMPOS_PREPARACION["Modelo base: copia a SSD local"]:.1f} s.')

if not modelo_sd21_valido(HYPIR_BASE_MODEL_LOCAL):
    raise RuntimeError('La caché SD 2.1 local no es válida después de la preparación.')

print('Modelo base SD 2.1 local:', HYPIR_BASE_MODEL_LOCAL)
print('HYPIR preparado. La caché maestra permanece en Drive y la inferencia carga SD 2.1 desde el SSD local.')

print('\nPreparación lista.')
print('Proyecto:', PROYECTO)
print('Repositorio CodeFormer:', CODEFORMER)
print('Entrada:', ENTRADA)
print('Resultados:', RESULTADOS)
if EN_COLAB:
    print('Pesos HYPIR/CodeFormer y caché Hugging Face persistentes en Drive.')
    print('Originales en:', DRIVE_ORIGINALES)
    print('Resultados en:', DRIVE_RESULTADOS)
try:
    import torch
    print('PyTorch:', torch.__version__)
    gpu_nombre = (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else 'no detectada'
    )
    print(
        'GPU disponible:', torch.cuda.is_available(), '-', gpu_nombre
    )
except Exception as e:
    print('No se pudo comprobar CUDA:', repr(e))
TIEMPOS_PREPARACION['Preparación total'] = time.perf_counter() - INICIO_PREPARACION
print('\n--- Tiempos de preparación por etapa ---')
for etapa, segundos in TIEMPOS_PREPARACION.items():
    print(f'{etapa}: {segundos:.1f} s')
print(f'Tiempo total de preparación: {TIEMPOS_PREPARACION["Preparación total"]:.1f} s')

def guardar_original_en_drive(origen):
    """Conserva una copia intacta del original en Drive, si está disponible."""
    origen = Path(origen)
    if EN_COLAB and 'DRIVE_ORIGINALES' in globals():
        destino = DRIVE_ORIGINALES / origen.name
        # No sobrescribir una copia anterior con el mismo nombre.
        if not destino.exists():
            shutil.copy2(origen, destino)
        return destino
    return origen

def liberar_memoria():
    """Libera basura de Python y caché CUDA que ya no está en uso."""
    import gc
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            # No usar ipc_collect en versiones antiguas de PyTorch.
            if hasattr(torch.cuda, 'ipc_collect'):
                torch.cuda.ipc_collect()
    except Exception as e:
        print('Aviso al liberar memoria:', repr(e))

def listar_imagenes():
    return sorted(
        [p for p in ENTRADA.iterdir()
         if p.is_file() and p.suffix.lower() in EXTENSIONES],
        key=lambda p: p.name.lower(),
    )


<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>📤 02 · Elegir una fotografía</h2>
</div>

Sube <strong>una imagen por vez</strong> o selecciona una fotografía que ya esté guardada en la carpeta de entrada.

Formatos habituales compatibles: JPG, JPEG, PNG, WEBP, BMP y TIFF.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

boton_subir = widgets.Button(
    description='Subir imagen',
    icon='upload',
    button_style='primary',
    layout=widgets.Layout(width='190px', height='40px'),
)
selector_imagen = widgets.Dropdown(
    options=[], description='Imagen:',
    layout=widgets.Layout(width='95%'),
)
estado_subida = widgets.Output(
    layout=widgets.Layout(
        width='100%',
        margin='14px 0 0 0',
        padding='12px 14px',
        border='1px solid #c9d9ea',
    )
)

def actualizar_selector(seleccion=None):
    imagenes = listar_imagenes()
    selector_imagen.options = [(p.name, str(p)) for p in imagenes]
    if seleccion and any(str(p) == seleccion for p in imagenes):
        selector_imagen.value = seleccion
    elif imagenes and selector_imagen.value not in [
        str(p) for p in imagenes
    ]:
        selector_imagen.value = str(imagenes[-1])
    elif not imagenes:
        selector_imagen.value = None

def subir_una_imagen(_):
    with estado_subida:
        clear_output()
        if not EN_COLAB:
            print(
                'La subida directa está disponible '
                'en Google Colab. '

                'Copia la imagen a:', ENTRADA
            )
            return
        carga = files.upload()
        if not carga:
            print('No se seleccionó ninguna imagen.')
            return
        # Si se eligen varios archivos, solo se guarda el primero.
        nombre, contenido = next(iter(carga.items()))
        ext = Path(nombre).suffix.lower()
        if ext not in EXTENSIONES:
            print(
                'Formato no compatible: '
                f'{ext or "(sin extensión)"}'
            )
            return
        destino = ENTRADA / Path(nombre).name
        destino.write_bytes(contenido)
        if 'DRIVE_ORIGINALES' in globals():
            shutil.copy2(destino, DRIVE_ORIGINALES / destino.name)
        actualizar_selector(str(destino))
        print('Imagen guardada:', destino)

boton_subir.on_click(subir_una_imagen)
actualizar_selector()
display(widgets.VBox(
    [
        widgets.HBox(
            [boton_subir],
            layout=widgets.Layout(margin='0 0 12px 0', gap='12px'),
        ),
        widgets.HTML(
            '<div style="margin:4px 0 10px;padding:10px 14px;'
            'border-radius:6px;background:#eef5ff;color:#203247;'
            'border-left:4px solid #2878c8;line-height:1.45">'
            '<b>Añadir una fotografía</b><br>'
            'Pulsa <b>Subir imagen</b> y, en el selector que aparece debajo, '
            'elige el archivo. Cuando termine la carga, podrás seleccionarlo '
            'en la lista de imágenes.</div>'
        ),
        selector_imagen,
        estado_subida,
    ],
    layout=widgets.Layout(width='100%', gap='8px', padding='4px 0 10px 0')
))


<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>🎚️ 03 · Ajustar la restauración</h2>
</div>

Elige el modelo en los controles de la celda siguiente y ajusta la ampliación.

- **HYPIR:** modelo generativo que puede reconstruir detalles que no estaban en la foto y dar un acabado artificial.
- **Fuerza HYPIR:** controla cuánto se mezcla el resultado generativo con el original ampliado con Lanczos. El perfil natural usa **0.35**: mezcla un 35 % del resultado generativo con un 65 % de la foto ampliada con Lanczos. Prueba **0.40–0.45** si quieres recuperar detalle; sube poco a poco, porque más fuerza puede volver el rostro artificial.
- **CodeFormer:** el control Fidelidad conserva su significado original. La mejora de rostros y del fondo sigue siendo opcional.
- **Ampliación 2× o 4×:** disponible para ambos motores; empieza por 2×.

💡 **Perfil natural predeterminado:** HYPIR a 2× con Fuerza HYPIR **0.35**, Fidelidad de CodeFormer **0.55** y las opciones de fondo y rostros activadas. Si el rostro parece demasiado suave, compara 0.40 y 0.45 con la misma imagen. Se guardarán tanto la imagen mezclada recomendada como el resultado HYPIR puro para compararlos. La mezcla reduce el aspecto artificial, pero no puede recuperar información que no existe en la foto.

**Memoria y archivos:** los pesos de HYPIR y CodeFormer y la caché del modelo base se guardan en Google Drive cuando está montado. Los originales se conservan en `originales/` y los resultados en `resultados/`. Cada ejecución procesa una imagen; al terminar se solicita limpieza de la memoria del proceso principal. La inferencia de los modelos se ejecuta en un proceso separado, que libera su memoria CUDA al finalizar.

**Aviso para imágenes exigentes:** 4× y las fotografías de gran tamaño pueden superar la memoria disponible. En Colab gratuito, empieza por 2× y procesa una imagen cada vez.

**Perfiles de procesamiento HYPIR:** por defecto se usa el perfil de calidad, con parches de 512 px y paso de 256 px (solapamiento de 256 px), para favorecer una restauración uniforme. También puedes activar **Modo rápido HYPIR**, que usa paso de 320 px (solapamiento de 192 px) y puede reducir el tiempo de procesamiento, con el riesgo de costuras o pérdida local de detalle. El modo rápido está desactivado por defecto; compara los resultados antes de usarlo en fotografías importantes. La caché distingue ambos perfiles y no reutiliza un resultado generado con un paso distinto.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

control_fidelidad = widgets.FloatSlider(
    value=0.55,
    min=0.0,
    max=1.0,
    step=0.05,
    description='Fidelidad:',
    readout_format='.2f',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%'),
)
control_mezcla_hypir = widgets.FloatSlider(
    value=0.35,
    min=0.35,
    max=1.00,
    step=0.05,
    description='Fuerza HYPIR:',
    readout_format='.2f',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%'),
)
control_escala = widgets.Dropdown(
    options=[('2×', 2), ('4×', 4)],
    value=2,
    description='Ampliación:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'),
)
control_hypir_rapido = widgets.Checkbox(
    value=False,
    description='Modo rápido HYPIR (menos solapamiento; puede afectar detalles locales)',
    indent=False,
    layout=widgets.Layout(width='100%'),
)
control_fondo = widgets.Checkbox(
    value=True,
    description='Mejorar también el fondo (Real-ESRGAN)',
    indent=False,
    layout=widgets.Layout(width='100%'),
)
control_rostros = widgets.Checkbox(
    value=True,
    description='Aplicar mejora adicional a los rostros',
    indent=False,
    layout=widgets.Layout(width='100%'),
)

panel_mejoras = widgets.VBox(
    [
        widgets.HTML(
            '<b>Opciones que tardan más (opcionales)</b>'
        ),
        control_fondo,
        control_rostros,
    ],
    layout=widgets.Layout(width='100%', gap='10px', padding='10px 12px', border='1px solid #d6e3f0'),
)
modelo_restauracion = widgets.Dropdown(
    options=[
        ('HYPIR · restauración generativa de detalles', 'HYPIR'),
        ('CodeFormer · restauración facial clásica', 'CodeFormer'),
    ],
    value='HYPIR',
    description='Modelo:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%'),
)
nota_modelo = widgets.HTML(
    '<div style="background:#eef5ff;border-left:4px solid #2878c8;padding:10px 14px;margin:8px 0 14px;border-radius:6px;color:#203247;line-height:1.5;">' 
    '<b>HYPIR:</b> reconstrucción generativa de detalles. La fuerza controla cuánto se combina su resultado con la foto ampliada original. ' 
    '<br><b>CodeFormer:</b> restauración facial clásica, con control de fidelidad.</div>'
)
display(modelo_restauracion)
display(nota_modelo)
display(control_fidelidad)
display(control_mezcla_hypir)
display(control_escala)
panel_rapido_hypir = widgets.VBox(
    [
        widgets.HTML(
            '<div style="background:#fff8e8;border-left:4px solid #d99a22;padding:9px 12px;border-radius:6px;line-height:1.45;">'
            '<b>Opcional · Modo rápido HYPIR</b><br>'
            'Reduce el solapamiento para procesar con menos parches. Puede introducir costuras o pérdida local de detalle. '
            'Desactivado por defecto para priorizar la calidad.</div>'
        ),
        control_hypir_rapido,
    ],
    layout=widgets.Layout(width='100%', gap='8px', padding='10px 12px', border='1px solid #ead9b5'),
)
if modelo_restauracion.value == 'HYPIR':
    display(panel_rapido_hypir)
display(panel_mejoras)


<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>🚀 04 · Restaurar la imagen</h2>
</div>

Cuando hayas elegido la foto y ajustado los parámetros, ejecuta la celda siguiente.

⏳ Espera a que aparezca el mensaje **«¡Restauración completada!»** antes de continuar.

**Optimización adicional:** en esta sesión, si repites exactamente la misma imagen con la misma ampliación y el mismo modo de solapamiento, HYPIR reutiliza la inferencia guardada en la caché local. Puedes cambiar la **Fuerza HYPIR** y volver a mezclar el resultado sin ejecutar de nuevo el modelo. La primera ejecución de cada combinación sigue funcionando como antes; la caché local se pierde al reiniciar el entorno.


In [ ]:
# ------------------------------------------------------------
# HYPIR: ejecutar el modelo alternativo mediante su inferencia oficial
# ------------------------------------------------------------
if 'modelo_restauracion' in globals() and modelo_restauracion.value == 'HYPIR':
    from datetime import datetime
    import time

    ultimo_resultado = None
    ultimo_original = None
    ultimo_directorio = None
    ultimo_modelo = 'HYPIR'

    if not selector_imagen.value:
        raise ValueError('Primero sube una imagen o selecciona una imagen en la sección de entrada.')
    origen = Path(selector_imagen.value)
    if not origen.is_file():
        raise FileNotFoundError(f'No existe la imagen seleccionada: {origen}')

    escala = int(control_escala.value)
    fuerza_hypir = float(control_mezcla_hypir.value)
    # Perfil de calidad por defecto: paso 256 (solapamiento 256 px).
    # Modo rápido opcional: paso 320 (solapamiento 192 px); puede reducir
    # el tiempo, pero aumenta el riesgo de costuras o pérdida local de detalle.
    modo_rapido_hypir = bool(
        'control_hypir_rapido' in globals() and control_hypir_rapido.value
    )
    stride_hypir = 320 if modo_rapido_hypir else 256
    guardar_original_en_drive(origen)
    marca = f'HYPIR_{origen.stem}_{escala}x_f{int(fuerza_hypir*100):02d}'
    out_dir = RESULTADOS / marca
    if out_dir.exists():
        marca += '_' + datetime.now().strftime('%Y%m%d_%H%M%S')
        out_dir = RESULTADOS / marca
    out_dir.mkdir(parents=True, exist_ok=True)

    # HYPIR admite JPG/PNG/BMP; convertir a PNG y normalizar orientación EXIF.
    entrada_hypir = PROYECTO / '_entrada_hypir'
    if entrada_hypir.exists():
        shutil.rmtree(entrada_hypir)
    entrada_hypir.mkdir(parents=True, exist_ok=True)
    imagen_hypir = entrada_hypir / (origen.stem + '.png')
    with Image.open(origen) as im:
        ImageOps.exif_transpose(im).convert('RGB').save(imagen_hypir, format='PNG')

    salida_hypir = PROYECTO / '_salida_hypir'
    if salida_hypir.exists():
        shutil.rmtree(salida_hypir)
    salida_hypir.mkdir(parents=True, exist_ok=True)

    comando_hypir = [
        sys.executable, str(HYPIR_SCRIPT),
        '--base_model_type', 'sd2',
        '--base_model_path', str(HYPIR_BASE_MODEL_LOCAL),
        '--model_t', '200', '--coeff_t', '200',
        '--lora_rank', '256',
        '--lora_modules', 'to_k,to_q,to_v,to_out.0,conv,conv1,conv2,conv_shortcut,conv_out,proj_in,proj_out,ff.net.2,ff.net.0.proj',
        '--weight_path', str(HYPIR_PESO),
        '--patch_size', '512', '--stride', str(stride_hypir),
        '--lq_dir', str(entrada_hypir),
        '--scale_by', 'factor', '--upscale', str(escala),
        '--captioner', 'empty',
        '--output_dir', str(salida_hypir),
        '--seed', '231', '--device', 'cuda' if __import__('torch').cuda.is_available() else 'cpu',
    ]

    with Image.open(origen) as im:
        ancho_entrada, alto_entrada = im.size
    megapixeles = ancho_entrada * alto_entrada / 1_000_000
    print('Modelo: HYPIR')
    print('Imagen:', origen.name)
    print(f'Resolución de entrada: {ancho_entrada} × {alto_entrada} ({megapixeles:.1f} MP)')
    print('Ampliación:', f'{escala}×')
    print(f'Fuerza HYPIR: {fuerza_hypir:.2f} (el resto se mezcla con la foto original ampliada)')
    perfil_hypir = 'rápido' if modo_rapido_hypir else 'calidad'
    print(f'Perfil HYPIR: {perfil_hypir}. Parches de 512 px, paso {stride_hypir} px (solapamiento {512 - stride_hypir} px).')
    if modo_rapido_hypir:
        print('Aviso: el modo rápido puede producir costuras o reducir detalles locales; revisa el resultado.')
    if megapixeles > 12:
        print('⚠️ Imagen grande: el procesamiento puede tardar más. Si falla por memoria, prueba una copia reducida.')
    print('\nProcesando con HYPIR… El modelo base se carga desde el SSD local; la primera preparación puede descargar varios GB.')
    # Caché segura de inferencias HYPIR puras. La huella se calcula sobre
    # la imagen RGB con orientación EXIF normalizada que realmente recibe HYPIR.
    # La fuerza de mezcla se excluye deliberadamente: se aplica después de la
    # inferencia y no cambia la salida generativa pura.
    import json
    cache_hypir_dir = PROYECTO / '_cache_resultados_hypir'
    cache_hypir_dir.mkdir(parents=True, exist_ok=True)
    huella_imagen_normalizada = calcular_sha256(imagen_hypir)
    parametros_cache_hypir = {
        'cache_schema': 2,
        'sha256_imagen_rgb_normalizada': huella_imagen_normalizada,
        'escala': escala,
        'patch_size': 512,
        'stride': stride_hypir,
        'seed': 231,
        'captioner': 'empty',
        'prompt': '',
        'model_t': 200,
        'coeff_t': 200,
        'lora_rank': 256,
        'lora_modules': 'to_k,to_q,to_v,to_out.0,conv,conv1,conv2,conv_shortcut,conv_out,proj_in,proj_out,ff.net.2,ff.net.0.proj',
        'base_model_id': HYPIR_BASE_MODEL_ID,
        'hypir_revision': HYPIR_REVISION,
        'hypir_peso_sha256': HYPIR_PESO_SHA256,
    }
    clave_cache_hypir = hashlib.sha256(
        json.dumps(parametros_cache_hypir, sort_keys=True).encode('utf-8')
    ).hexdigest()
    archivo_cache_hypir = cache_hypir_dir / f'{clave_cache_hypir}.png'
    ruta_hypir_esperada = salida_hypir / 'result' / (origen.stem + '.png')
    ruta_hypir_esperada.parent.mkdir(parents=True, exist_ok=True)
    cache_hypir_utilizada = False
    with Image.open(imagen_hypir) as im:
        ancho_normalizado, alto_normalizado = im.size
    dimensiones_esperadas = (ancho_normalizado * escala, alto_normalizado * escala)

    inicio_proceso = time.perf_counter()
    if archivo_cache_hypir.is_file() and archivo_cache_hypir.stat().st_size > 0:
        try:
            with Image.open(archivo_cache_hypir) as im:
                im.verify()
            # verify() comprueba integridad del archivo, no sus dimensiones.
            # Comprobar también que coincida la escala esperada antes de reutilizar.
            with Image.open(archivo_cache_hypir) as im:
                if im.size != dimensiones_esperadas:
                    raise ValueError(
                        f'dimensiones de caché {im.size}; se esperaban {dimensiones_esperadas}'
                    )
                if im.format != 'PNG':
                    raise ValueError(f'formato de caché inesperado: {im.format}')
            shutil.copy2(archivo_cache_hypir, ruta_hypir_esperada)
            cache_hypir_utilizada = True
            registro = (
                'CACHÉ HYPIR VALIDADA: coinciden la huella de la imagen normalizada, '
                'los parámetros/modelos y las dimensiones de salida. '
                'Se omiten la carga de modelos y la inferencia.'
            )
            proceso = subprocess.CompletedProcess(
                args=comando_hypir, returncode=0, stdout=registro
            )
        except Exception as error_cache:
            print('Aviso: caché HYPIR no válida; se eliminará y recalculará:', repr(error_cache))
            archivo_cache_hypir.unlink(missing_ok=True)

    # Ejecutar HYPIR dentro del kernel para conservar
    # SD2Enhancer en memoria entre imágenes distintas de esta misma sesión.
    # La caché de inferencia existente sigue teniendo prioridad.
    tiempo_carga_hypir = 0.0
    if not cache_hypir_utilizada:
        import contextlib
        import io
        import torch
        # El script oficial se ejecutaba con cwd=HYPIR_REPO; al importar dentro
        # del kernel, añadimos explícitamente el repositorio al import path.
        ruta_repo_hypir = str(HYPIR_REPO)
        if ruta_repo_hypir not in sys.path:
            sys.path.insert(0, ruta_repo_hypir)
        from accelerate.utils import set_seed
        from torchvision import transforms
        from HYPIR.enhancer.sd2 import SD2Enhancer

        firma_modelo_hypir = (
            str(HYPIR_BASE_MODEL_LOCAL), str(HYPIR_PESO), HYPIR_PESO_SHA256,
            HYPIR_REVISION, 'sd2', 200, 200, 256,
            'to_k,to_q,to_v,to_out.0,conv,conv1,conv2,conv_shortcut,conv_out,proj_in,proj_out,ff.net.2,ff.net.0.proj',
            'cuda' if torch.cuda.is_available() else 'cpu',
        )
        modelo_hypir_memoria = globals().get('_HYPIR_MODELO_EN_MEMORIA')
        firma_previa = globals().get('_HYPIR_FIRMA_MODELO_EN_MEMORIA')
        if modelo_hypir_memoria is None or firma_previa != firma_modelo_hypir:
            # Si cambió la configuración, liberar la referencia anterior antes de recargar.
            if modelo_hypir_memoria is not None:
                del modelo_hypir_memoria
                globals().pop('_HYPIR_MODELO_EN_MEMORIA', None)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            print('Cargando HYPIR en memoria para reutilizarlo en esta sesión…')
            inicio_carga_memoria = time.perf_counter()
            modelo_hypir_memoria = SD2Enhancer(
                base_model_path=str(HYPIR_BASE_MODEL_LOCAL),
                weight_path=str(HYPIR_PESO),
                lora_modules=firma_modelo_hypir[8].split(','),
                lora_rank=256, model_t=200, coeff_t=200,
                device=firma_modelo_hypir[9],
            )
            modelo_hypir_memoria.init_models()
            tiempo_carga_hypir = time.perf_counter() - inicio_carga_memoria
            globals()['_HYPIR_MODELO_EN_MEMORIA'] = modelo_hypir_memoria
            globals()['_HYPIR_FIRMA_MODELO_EN_MEMORIA'] = firma_modelo_hypir
            print(f'Modelos HYPIR cargados y conservados en memoria: {tiempo_carga_hypir:.2f} s')
        else:
            print('HYPIR: reutilizando los modelos que siguen cargados en memoria.')

        # Mantener la semilla oficial y el mismo método de inferencia de test.py.
        set_seed(231)
        with Image.open(imagen_hypir) as im:
            imagen_pil_hypir = im.convert('RGB')
        tensor_hypir = transforms.ToTensor()(imagen_pil_hypir).unsqueeze(0)
        registro_buffer = io.StringIO()
        inicio_inferencia_memoria = time.perf_counter()
        try:
            with contextlib.redirect_stdout(registro_buffer):
                resultado_pil_hypir = modelo_hypir_memoria.enhance(
                    lq=tensor_hypir,
                    prompt='',
                    scale_by='factor',
                    upscale=escala,
                    patch_size=512,
                    stride=stride_hypir,
                    return_type='pil',
                )[0]
            resultado_pil_hypir.save(ruta_hypir_esperada, format='PNG')
            tiempo_inferencia_memoria = time.perf_counter() - inicio_inferencia_memoria
            registro = (
                f'Modelos en memoria GPU reutilizables. Carga en esta ejecución: '
                f'{tiempo_carga_hypir:.2f} s. Inferencia: {tiempo_inferencia_memoria:.2f} s.\n'
                + registro_buffer.getvalue()
            )
            proceso = subprocess.CompletedProcess(args=comando_hypir, returncode=0, stdout=registro)
        except Exception:
            # No dejar una salida parcial que pueda confundirse con una inferencia válida.
            ruta_hypir_esperada.unlink(missing_ok=True)
            raise
        finally:
            del tensor_hypir, imagen_pil_hypir
    tiempo_proceso_hypir = time.perf_counter() - inicio_proceso
    print('\n--- Registro de HYPIR ---')
    print(registro[-18000:] if registro.strip() else '(sin salida de texto)')
    # En V4 la carga/inferencia se cronometra directamente en el kernel.
    tiempo_carga_hypir = float(tiempo_carga_hypir or 0.0)
    if not cache_hypir_utilizada:
        print(f'Tiempo de carga de modelos HYPIR en esta ejecución: {tiempo_carga_hypir:.2f} s')
    print(f'Tiempo total del proceso HYPIR (carga + inferencia + pre/postprocesado): {tiempo_proceso_hypir:.2f} s')
    if not cache_hypir_utilizada:
        print('Tiempo de inferencia y operaciones auxiliares: '
              f'{max(0.0, tiempo_proceso_hypir - tiempo_carga_hypir):.2f} s')
    if proceso.returncode != 0:
        if proceso.returncode == -9:
            raise RuntimeError('HYPIR fue terminado por falta de memoria. Prueba con una imagen más pequeña o una sesión con más RAM/GPU.')
        raise RuntimeError(f'HYPIR terminó con código {proceso.returncode}. Consulta el registro anterior.')

    ruta_hypir = salida_hypir / 'result' / (origen.stem + '.png')
    if not ruta_hypir.is_file() or ruta_hypir.stat().st_size == 0:
        raise FileNotFoundError(f'HYPIR terminó sin crear la imagen esperada: {ruta_hypir}')
    with Image.open(ruta_hypir) as im:
        im.verify()
    with Image.open(ruta_hypir) as im:
        if im.size != dimensiones_esperadas:
            raise ValueError(
                f'HYPIR generó dimensiones {im.size}, pero se esperaban '
                f'{dimensiones_esperadas}; no se guardará en caché.'
            )
        if im.format != 'PNG':
            raise ValueError(f'HYPIR generó un formato inesperado: {im.format}')

    # Guardar la inferencia pura tras verificarla. Si la misma imagen se vuelve
    # a procesar con la misma escala/stride, se reutiliza; cambiar la fuerza de
    # mezcla no invalida esta caché. El caché es local a esta sesión de Colab.
    if not cache_hypir_utilizada:
        temporal_cache_hypir = archivo_cache_hypir.with_suffix('.png.tmp')
        try:
            shutil.copy2(ruta_hypir, temporal_cache_hypir)
            temporal_cache_hypir.replace(archivo_cache_hypir)
            print('Inferencia pura guardada en caché local para reutilizarla en esta sesión.')
        except Exception as error_guardado_cache:
            temporal_cache_hypir.unlink(missing_ok=True)
            print('Aviso: no se pudo guardar la caché HYPIR; el resultado sigue siendo válido:', repr(error_guardado_cache))
    else:
        print('Caché HYPIR reutilizada: no se cargaron modelos ni se repitió la inferencia.')

    carpeta_final = out_dir / 'final_results'
    carpeta_final.mkdir(parents=True, exist_ok=True)

    # Conservar también el resultado generativo puro para poder compararlo.
    ruta_hypir_puro = carpeta_final / f'{origen.stem}_HYPIR_sin_mezcla.png'
    shutil.copy2(ruta_hypir, ruta_hypir_puro)

    # Mezclar HYPIR con el original ampliado por Lanczos.
    # Menor fuerza = más fidelidad visual y menos detalles inventados.
    with Image.open(origen) as im:
        original_rgb = ImageOps.exif_transpose(im).convert('RGB')
    with Image.open(ruta_hypir) as im:
        restaurada_rgb = im.convert('RGB')
    base_ampliada = original_rgb.resize(
        restaurada_rgb.size, Image.Resampling.LANCZOS
    )
    resultado_mezclado = Image.blend(
        base_ampliada, restaurada_rgb, alpha=fuerza_hypir
    )
    ultimo_resultado = carpeta_final / f'{origen.stem}.png'
    resultado_mezclado.save(ultimo_resultado, format='PNG')

    ultimo_original = origen
    ultimo_directorio = out_dir
    guardar_original_en_drive(origen)

    if 'DRIVE_RESULTADOS' in globals():
        carpeta_drive = DRIVE_RESULTADOS / marca
        carpeta_drive.mkdir(parents=True, exist_ok=True)
        copia_drive = carpeta_drive / ultimo_resultado.name
        shutil.copy2(ultimo_resultado, copia_drive)
        shutil.copy2(ruta_hypir_puro, carpeta_drive / ruta_hypir_puro.name)
        print('Copia permanente en Drive:', copia_drive)
        print('También se guardó el HYPIR puro para comparar:', carpeta_drive / ruta_hypir_puro.name)

    with Image.open(origen) as im:
        dimensiones_originales = ImageOps.exif_transpose(im).size
    with Image.open(ultimo_resultado) as im:
        dimensiones_restauradas = im.size
    print('\n¡Restauración completada!')
    print('Resultado mezclado (recomendado):', ultimo_resultado)
    print('HYPIR sin mezcla:', ruta_hypir_puro)
    print('Dimensiones originales:', dimensiones_originales)
    print('Dimensiones restauradas:', dimensiones_restauradas)
    print(f'Tiempo total del proceso HYPIR confirmado: {tiempo_proceso_hypir:.1f} s')
    # V4 conserva deliberadamente el objeto del modelo en memoria entre imágenes.
    # No llamar a liberar_memoria() aquí: vaciaría cachés CUDA útiles para la sesión.
    del registro, proceso
    if cache_hypir_utilizada:
        liberar_memoria()

else:

    from datetime import datetime
    import time

    ultimo_resultado = None
    ultimo_original = None
    ultimo_directorio = None

    # ------------------------------------------------------------
    # 1. Validar la imagen y leer los controles
    # ------------------------------------------------------------
    if not selector_imagen.value:
        raise ValueError(
            'Primero sube una imagen o selecciona una imagen '
            'en la sección de entrada.'
        )

    origen = Path(selector_imagen.value)

    if not origen.is_file():
        raise FileNotFoundError(
            f'No existe la imagen seleccionada: {origen}'
        )

    fidelidad = float(control_fidelidad.value)
    escala = int(control_escala.value)
    guardar_original_en_drive(origen)
    porcentaje = int(round(fidelidad * 100))

    marca = (
        f'restauracion_{origen.stem}_w{porcentaje:02d}_{escala}x'
    )

    out_dir = RESULTADOS / marca

    if out_dir.exists():
        marca += '_' + datetime.now().strftime('%Y%m%d_%H%M%S')
        out_dir = RESULTADOS / marca

    out_dir.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------
    # 2. Preparar la imagen de entrada para CodeFormer
    # La rama predeterminada utiliza los argumentos -i y -o.
    # Se conserva una copia PNG temporal para admitir los formatos
    # de imagen que Pillow puede abrir.
    # ------------------------------------------------------------
    carpeta_entrada_codeformer = (
        PROYECTO / '_entrada_codeformer' / origen.name
    )

    if carpeta_entrada_codeformer.exists():
        shutil.rmtree(carpeta_entrada_codeformer)

    carpeta_entrada_codeformer.mkdir(
        parents=True,
        exist_ok=True,
    )

    # CodeFormer busca archivos JPG/PNG con un patrón glob.
    # Convertimos la imagen seleccionada a PNG para admitir
    # también entradas JPEG, WEBP, TIFF y otros formatos compatibles
    # con Pillow.
    nombre_entrada_codeformer = origen.stem + '.png'
    imagen_entrada_codeformer = (
        carpeta_entrada_codeformer / nombre_entrada_codeformer
    )

    with Image.open(origen) as im:
        # Aplicar la orientación EXIF físicamente antes de convertir a PNG.
        # Así CodeFormer recibe la fotografía en la orientación correcta.
        ImageOps.exif_transpose(im).convert('RGB').save(
            imagen_entrada_codeformer,
            format='PNG',
        )

    # Se indica explícitamente la carpeta de salida con -o.
    carpeta_resultados_codeformer = (
        CODEFORMER / 'results' / f'{origen.stem}_w{porcentaje:02d}_{escala}x'
    )

    # Borrar únicamente el resultado previo de esta misma ejecución
    # lógica para evitar confundir archivos antiguos con el nuevo.
    if carpeta_resultados_codeformer.exists():
        shutil.rmtree(carpeta_resultados_codeformer)

    # ------------------------------------------------------------
    # 3. Construir el comando
    # ------------------------------------------------------------
    comando = [
        sys.executable,
        str(SCRIPT),
        '-i',
        str(imagen_entrada_codeformer),
        '-o',
        str(carpeta_resultados_codeformer),
        '-w',
        str(fidelidad),
        '-s',
        str(escala),
    ]

    if control_fondo.value:
        comando.extend([
            '--bg_upsampler',
            'realesrgan',
        ])

    if control_rostros.value:
        comando.append('--face_upsample')

    # ------------------------------------------------------------
    # 4. Mostrar la configuración
    # ------------------------------------------------------------
    print('Imagen:', origen.name)
    print('Fidelidad:', fidelidad)
    print('Ampliación:', f'{escala}×')
    print(
        'Mejora del fondo:',
        'activada' if control_fondo.value else 'desactivada',
    )
    print(
        'Restauración facial:',
        'activada' if control_rostros.value else 'desactivada',
    )

    with Image.open(origen) as im:
        ancho_entrada, alto_entrada = im.size

    megapixeles = (
        ancho_entrada * alto_entrada
    ) / 1_000_000

    print(
        f'Resolución de entrada: {ancho_entrada} × '
        f'{alto_entrada} ({megapixeles:.1f} MP)'
    )

    if escala == 4 and megapixeles >= 20:
        print(
            '⚠️ AVISO: la imagen es grande para ampliación 4× '
            'y puede superar la memoria disponible.'
        )
    elif escala == 4 and control_fondo.value and control_rostros.value:
        print(
            '⚠️ AVISO: la ampliación 4× con las dos mejoras '
            'puede consumir mucha memoria.'
        )

    # ------------------------------------------------------------
    # 5. Ejecutar CodeFormer
    # ------------------------------------------------------------
    print('\nProcesando…')
    print('Carpeta de entrada:', carpeta_entrada_codeformer)
    print('Salida esperada:', carpeta_resultados_codeformer)

    inicio_proceso = time.time()

    proceso = subprocess.run(
        comando,
        cwd=str(CODEFORMER),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    registro = proceso.stdout or ''
    print('\n--- Registro de CodeFormer ---')
    print(registro if registro.strip() else '(sin salida de texto)')

    if proceso.returncode != 0:
        if proceso.returncode == -9:
            raise RuntimeError(
                'CodeFormer fue terminado por el sistema (código -9), '
                'normalmente por falta de memoria. Prueba la ampliación 2×.'
            )

        raise RuntimeError(
            f'CodeFormer terminó con código {proceso.returncode}. '
            'Consulta el registro anterior.'
        )

    # ------------------------------------------------------------
    # 6. Localizar el resultado esperado
    # La rama predeterminada guarda la imagen directamente en la
    # carpeta indicada mediante -o.
    # ------------------------------------------------------------
    # CodeFormer guarda la imagen completa dentro de final_results/.
    ruta_final_codeformer = (
        carpeta_resultados_codeformer / 'final_results' / f'{origen.stem}.png'
    )

    if (
        not ruta_final_codeformer.is_file()
        or ruta_final_codeformer.stat().st_size == 0
    ):
        # Diagnóstico específico: mostrar qué ha creado realmente
        # el script, en vez de atribuirle una salida inexistente.
        archivos_salida = []

        if carpeta_resultados_codeformer.exists():
            archivos_salida = [
                p for p in carpeta_resultados_codeformer.rglob('*')
                if p.is_file()
            ]

        print('\n--- Diagnóstico de salida ---')
        print('Código de retorno:', proceso.returncode)
        print('Carpeta esperada:', carpeta_resultados_codeformer)
        print('Existe la carpeta:', carpeta_resultados_codeformer.exists())

        if archivos_salida:
            for p in archivos_salida[:50]:
                print(f'{p} ({p.stat().st_size:,} bytes)')
        else:
            print('CodeFormer no ha creado archivos en la carpeta esperada.')

        raise FileNotFoundError(
            'CodeFormer terminó sin producir la imagen final esperada. '
            'El registro y el listado anterior permiten distinguir '
            'un problema de detección facial de un problema de guardado.'
        )

    # Verificar que el resultado se puede abrir como imagen.
    try:
        with Image.open(ruta_final_codeformer) as im:
            im.verify()
    except Exception as e:
        raise RuntimeError(
            f'El archivo de salida existe, pero no es una imagen válida: '
            f'{ruta_final_codeformer}. Detalle: {e}'
        ) from e

    # ------------------------------------------------------------
    # 7. Copiar a la carpeta de resultados del proyecto
    # ------------------------------------------------------------
    carpeta_final = out_dir / 'final_results'
    carpeta_final.mkdir(parents=True, exist_ok=True)

    ultimo_resultado = carpeta_final / ruta_final_codeformer.name
    shutil.copy2(ruta_final_codeformer, ultimo_resultado)

    ultimo_original = origen
    ultimo_directorio = out_dir

    # Guardar una copia permanente en Google Drive, si está configurado.
    if 'DRIVE_RESULTADOS' in globals():
        carpeta_drive = DRIVE_RESULTADOS / marca
        carpeta_drive.mkdir(parents=True, exist_ok=True)

        copia_drive = carpeta_drive / ultimo_resultado.name
        shutil.copy2(ultimo_resultado, copia_drive)

        print('\nCopia permanente en Drive:', copia_drive)

    with Image.open(origen) as im:
        dimensiones_originales = im.size

    with Image.open(ultimo_resultado) as im:
        dimensiones_restauradas = im.size

    print('\n¡Restauración completada!')
    print('Resultado:', ultimo_resultado)
    print('Dimensiones originales:', dimensiones_originales)
    print('Dimensiones restauradas:', dimensiones_restauradas)


<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>🖼️ 05 · Comparar el antes y el después</h2>
</div>

Revisa las dos imágenes una al lado de la otra para comprobar los cambios y la calidad del resultado.

In [ ]:


from PIL import Image, ImageOps
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from io import BytesIO
import base64

if (
    'ultimo_resultado' not in globals()
    or ultimo_resultado is None
    or not Path(ultimo_resultado).is_file()
):
    print(
        'Todavía no hay una restauración disponible. '
        'Ejecuta primero la celda de restauración.'
    )
else:
    # Cargar las imágenes respetando su orientación EXIF.
    with Image.open(ultimo_original) as im:
        original_rgb = ImageOps.exif_transpose(im).convert('RGB').copy()

    with Image.open(ultimo_resultado) as im:
        restaurada_rgb = ImageOps.exif_transpose(im).convert('RGB').copy()

    # Crear la comparación.
    fig, axes = plt.subplots(1, 2, figsize=(14, 8))

    axes[0].imshow(original_rgb)
    axes[0].set_title(
        f'Original ({original_rgb.width} × {original_rgb.height})'
    )
    axes[0].axis('off')

    axes[1].imshow(restaurada_rgb)
    axes[1].set_title(
        f'{globals().get("ultimo_modelo", "Restaurada")} ({restaurada_rgb.width} × {restaurada_rgb.height})'
    )
    axes[1].axis('off')

    fig.tight_layout(pad=2)

    # Convertir la figura a PNG en memoria.
    buffer = BytesIO()
    fig.savefig(
        buffer,
        format='png',
        dpi=120,
        bbox_inches='tight',
        facecolor='white'
    )
    plt.close(fig)

    imagen_base64 = base64.b64encode(
        buffer.getvalue()
    ).decode('utf-8')

    # Centrar la figura completa dentro de la salida de Colab.
    display(HTML(f"""
    <div style="
        width: 100%;
        display: flex;
        justify-content: center;
        align-items: center;
        box-sizing: border-box;
    ">
        <img
            src="data:image/png;base64,{imagen_base64}"
            style="
                display: block;
                max-width: 100%;
                height: auto;
                margin: 0 auto;
            "
        />
    </div>
    """))

    print(
        'Resultado listo para descargar:',
        Path(ultimo_resultado).name
    )



<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>📥 06 · Descargar la imagen restaurada</h2>
</div>

Ejecuta esta celda cuando estés conforme con el resultado. La imagen restaurada también se conserva en la carpeta de resultados de tu proyecto en Google Drive.

✅ Si todavía no hay una restauración disponible, vuelve a la sección **04 · Restaurar la imagen**.

In [ ]:
if (
    'ultimo_resultado' not in globals()
    or ultimo_resultado is None
    or not Path(ultimo_resultado).is_file()
):
    print(
        'No hay una imagen restaurada para descargar. '
        'Ejecuta primero la restauración y la comparación.'
    )
elif not EN_COLAB:
    print(
        'La imagen restaurada está guardada en:',
        ultimo_resultado,
    )
else:
    print('Descargando:', Path(ultimo_resultado).name)
    files.download(str(ultimo_resultado))
